# Mass Spectrometry Analysis of Extracellular Vesicles Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Use the `@id` for each entity as reference.

We'll print all record sets, their fields, and column keys.

In [ ]:
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (field @id: {field.id})")
        if hasattr(field, 'columns'):
            print("      Columns:")
            for col in field.columns:
                print(f"        - {col.name} (column @id: {col.id})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We'll select the primary record set containing protein abundance data.

**Note:** For this dataset, the main data record set is named `EV_MS_Proteins` with `@id` equal to `'cr:EV_MS_Proteins'`. Please verify in the overview above for the exact name and fields in case of updates.


In [ ]:
# --- Replace with your chosen record set(s) based on the printed output above ---
record_sets = ['cr:EV_MS_Proteins']
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records.")
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

We'll select a numeric field, such as `cr:MW` (molecular weight), and filter out proteins with MW < 10000, normalize that field, and if applicable, group by another field such as `cr:Accession` or `cr:Experiment`. 

**Edit the variable assignments below as appropriate for your dataset. See field `@id` from the overview above.**

In [ ]:
# Choose your IDs
record_set_id = 'cr:EV_MS_Proteins'
numeric_field_id = 'cr:MW'           # i.e., Molecular Weight
group_field_id = 'cr:Sample_Group'   # e.g. Group/Experiment if available

# Check available DataFrame and columns
df = dataframes[record_set_id]
print('Columns available:', df.columns.tolist())

# Ensure numeric field is float (may require cleaning if loaded as object/string)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter records with MW > 10000
threshold = 10000
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
display(filtered_df[[numeric_field_id]].head())

# Normalize MW
filtered_df[numeric_field_id + '_normalized'] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Group by field if present
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'std', 'count'])
    print(f"Grouped mean/std/count by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. Let's plot the filtered and normalized MW (molecular weight) distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of Molecular Weight
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], bins=40, kde=True)
plt.xlabel('Molecular Weight (MW)')
plt.title('Distribution of Protein Molecular Weights (MW)')
plt.show()

# Boxplot by Group Field (if available)
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.ylabel('Molecular Weight (MW)')
    plt.title('MW Distribution by Sample Group')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 extracellular vesicle mass spectrometry dataset using mlcroissant. We reviewed the data schema, loaded records by `@id`, filtered and normalized protein MW values, and visualized MW distributions.

**Next steps:**
- Investigate other relevant fields (e.g., peptide counts, abundance measures) using their `@id`s
- Apply additional filtering and group analysis
- Use this notebook as a reproducible template for FAIR Croissant-compliant datasets!